# Understanding the Tables — How Phases 1 → 3 Fit Together

This notebook is a **read-only tour**. It doesn't build anything new — it just
loads the tables that Phases 1–3 already produced and shows you *what each one
looks like*, *what it's for*, and *how they join together*.

**Why are there so many tables?** Because each stage of the pipeline refines the
data one step further. You can't jump from messy raw JSON straight to ratings —
you build a chain of trustworthy tables, each one the clean input to the next:

```
raw_data/  ->  master tables  ->  enriched events  ->  player_match_stats (atomic)
```

| Table | Grain (1 row = ) | Built in | Purpose |
|---|---|---|---|
| `players_master`    | one player        | Phase 2 | who the player is (bio + position) |
| `teams_master`      | one team          | Phase 2 | team id -> name |
| `matches_master`    | one match         | Phase 2 | scores, coaches, ET/pen flags |
| `player_appearances`| one player-match  | Phase 2 | **minutes played** + authoritative goals/cards |
| `events_enriched`   | one event         | Phase 3 | every on-ball action, flagged |
| `player_match_stats`| one player-match  | Phase 3 | **THE atomic table**: minutes + all event stats joined |


## Setup — load every table


In [1]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import wyscout_lib as wl

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 240)
DATA = wl.DATA

players     = pd.read_parquet(DATA / 'players_master.parquet')
teams       = pd.read_parquet(DATA / 'teams_master.parquet')
matches     = pd.read_parquet(DATA / 'matches_master.parquet')
appearances = pd.read_parquet(DATA / 'player_appearances.parquet')
pms         = pd.read_parquet(DATA / 'player_match_stats.parquet')
events_wc   = pd.read_parquet(DATA / 'events_enriched' / 'world_cup.parquet')

summary = pd.DataFrame([
    ['players_master',     *players.shape],
    ['teams_master',       *teams.shape],
    ['matches_master',     *matches.shape],
    ['player_appearances', *appearances.shape],
    ['events_enriched/WC', *events_wc.shape],
    ['player_match_stats', *pms.shape],
], columns=['table', 'rows', 'columns'])
summary


,table,rows,columns
0,players_master,3603,11
1,teams_master,142,5
2,matches_master,1941,16
3,player_appearances,74098,11
4,events_enriched/WC,101759,43
5,player_match_stats,53380,42


## Phase 1 (EDA) — no table, just the rulebook

Phase 1 produces **no data table**. It audits the raw files and proves the
rules the rest of the code obeys (goals come from the match sheet, coordinates
are team-relative, cards are minutes, etc.). Those decisions are baked into
`wyscout_lib.py` as constants. Quick proof they're loaded:


In [2]:
print('Competitions:', list(wl.COMPETITIONS))
print('Positions    :', wl.POSITIONS)
print('Extra-time periods:', wl.EXTRA_TIME_PERIODS, '| Shootout:', wl.SHOOTOUT_PERIOD)
print('Goal tag:', wl.TAG_GOAL, '| Accurate tag:', wl.TAG_ACCURATE, '| Assist tag:', wl.TAG_ASSIST)


Competitions: ['england', 'france', 'germany', 'italy', 'spain', 'euro_2016', 'world_cup']
Positions    : ['GK', 'DF', 'MD', 'FW']
Extra-time periods: {'E1', 'E2'} | Shootout: P
Goal tag: 101 | Accurate tag: 1801 | Assist tag: 301


## 1. `players_master`  —  one row per player

Straight from `players.json`. The key column is **`position_code`**
(GK/DF/MD/FW) — it decides which peer group a player is ranked against later.
Names were repaired from double-escaped unicode (Phase 1 trap).


In [3]:
print(players.dtypes)
players.head(3)


player_id          int64
first_name           str
last_name            str
short_name           str
birth_date           str
nationality          str
position_code        str
height_cm          Int64
weight_kg          Int64
preferred_foot       str
current_team_id    Int64
dtype: object


,player_id,first_name,last_name,short_name,birth_date,nationality,position_code,height_cm,weight_kg,preferred_foot,current_team_id
0,32777,Harun,Tekin,H. Tekin,1989-06-17,Turkey,GK,187,78,right,4502
1,393228,Malang,Sarr,M. Sarr,1999-01-23,Senegal,DF,182,73,left,3775
2,393230,Over,Mandanda,O. Mandanda,1998-10-26,France,GK,176,72,NaN,3772


## 2. `teams_master`  —  one row per team

Only really used to turn a `team_id` into a readable name on the coach cards.


In [4]:
teams.head(3)


,team_id,team_name,official_name,team_type,area
0,1613,Newcastle United,Newcastle United FC,club,England
1,692,Celta de Vigo,Real Club Celta de Vigo,club,Spain
2,691,Espanyol,Reial Club Deportiu Espanyol,club,Spain


## 3. `matches_master`  —  one row per match

Scores, both coach ids, and the ET/penalty flags. `home_coach_id` /
`away_coach_id` are the seed for Phase 7 (coach ratings). Notice `has_extra_time`
— that's what decided whether minutes were capped at 90 or 120.


In [5]:
print('matches:', len(matches),
      '| extra-time:', int(matches.has_extra_time.sum()),
      '| penalty shootouts:', int(matches.has_penalties.sum()))
matches[['match_id','competition_id','label','home_score','away_score',
         'winner_team_id','has_extra_time','home_coach_id','away_coach_id']].head(3)


matches: 1941 | extra-time: 10 | penalty shootouts: 7


,match_id,competition_id,label,home_score,away_score,winner_team_id,has_extra_time,home_coach_id,away_coach_id
0,2500089,england,"Burnley - AFC Bournemouth, 1 - 2",1,2,1659.0,False,8880.0,8934.0
1,2500090,england,"Crystal Palace - West Bromwich Albion, 2 - 0",2,0,1628.0,False,8357.0,NaN
2,2500091,england,"Huddersfield Town - Arsenal, 0 - 1",0,1,1609.0,False,18572.0,7845.0


## 4. `player_appearances`  —  one row per (player, match)

The output of `build_matches_and_appearances`. The star column is
**`minutes_played`** — the denominator of every per-90 stat later. It includes
unused subs as 0-minute rows (they get filtered out in the next step).
`goals` here is the **authoritative** match-sheet count; cards are 0/1 flags.


In [6]:
print('total appearance rows :', len(appearances))
print('actually played (>0m) :', int((appearances.minutes_played>0).sum()))
print('unused subs (0 min)   :', int((appearances.minutes_played==0).sum()))
appearances.head(5)


total appearance rows : 74098
actually played (>0m) : 53380
unused subs (0 min)   : 20718


,player_id,match_id,competition_id,team_id,started,minutes_played,subbed_on_minute,subbed_off_minute,goals,yellow_card,red_card
0,9206,2500089,england,1646,True,61.0,NaN,61.0,1,0,0
1,93,2500089,england,1646,True,80.0,NaN,80.0,0,0,0
2,10108,2500089,england,1646,True,90.0,NaN,NaN,0,0,0
3,8433,2500089,england,1646,True,90.0,NaN,NaN,0,0,0
4,8125,2500089,england,1646,True,90.0,NaN,NaN,0,0,0


## 5. `events_enriched/<comp>.parquet`  —  one row per event

The output of `enrich_events`. Every raw on-ball action becomes a row with
~30 boolean/numeric **flags** pre-computed (so the next step is a fast groupby).
Below: a few passes from the World Cup showing the flag columns.


In [7]:
print('World Cup events:', len(events_wc))
cols = ['event_id','player_id','event_name','sub_event_name','start_x','end_x',
        'is_pass','pass_accurate','is_progressive_pass','is_shot','shot_on_target','event_goal']
events_wc[events_wc.is_pass].head(4)[cols]


World Cup events: 101759


,event_id,player_id,event_name,sub_event_name,start_x,end_x,is_pass,pass_accurate,is_progressive_pass,is_shot,shot_on_target,event_goal
0,258612104,122671,Pass,Simple pass,50,35,True,True,False,False,False,False
1,258612106,139393,Pass,High pass,35,75,True,True,True,False,False,False
4,258612110,122847,Pass,Simple pass,63,71,True,True,False,False,False,False
5,258612113,122832,Pass,Simple pass,71,92,True,True,True,False,False,False


## 6. `player_match_stats`  —  THE atomic table (the join)

The output of `build_player_match_stats`. This is where the pipeline's **first
real join** happens:

```
player_appearances  (minutes + authoritative goals)   <-- the 'spine'
        +
events aggregated per (match, player)  (passes, shots, duels, ...)
        =
player_match_stats   (everything about one player in one match)
```

It keeps BOTH `goals` (authoritative, from the match sheet) and `event_goals`
(counted from event tag 101) side by side — so we can prove they agree.


In [8]:
print('rows:', len(pms), '| columns:', pms.shape[1])
pms[['player_id','match_id','minutes_played','goals','event_goals','assists',
     'pass_total','pass_accurate','shots_total','shots_on_target','duels_total']].head(5)


rows: 53380 | columns: 42


,player_id,match_id,minutes_played,goals,event_goals,assists,pass_total,pass_accurate,shots_total,shots_on_target,duels_total
0,9206,2500089,61.0,1,1,0,13,7,1,1,17
1,93,2500089,80.0,0,0,0,27,21,0,0,7
2,10108,2500089,90.0,0,0,0,45,39,1,0,19
3,8433,2500089,90.0,0,0,0,47,23,0,0,13
4,8125,2500089,90.0,0,0,0,45,41,1,0,18


### Proof the join is sound: authoritative vs event goals

Two totally different sources, so ~100% agreement means the goal logic is right.


In [9]:
g_auth, g_evt = int(pms.goals.sum()), int(pms.event_goals.sum())
print(f'authoritative goals: {g_auth:,}')
print(f'event-derived goals: {g_evt:,}')
print(f'agreement          : {100*g_evt/g_auth:.1f}%')


authoritative goals: 5,049
event-derived goals: 5,048
agreement          : 100.0%


## 7. See the join happen for ONE player-match

Let's take Harry Kane's biggest World Cup game and rebuild his one
`player_match_stats` row from its two ingredients: his appearance row
(minutes + goals) and his aggregated events (passes, shots).


In [10]:
kane_id = players.loc[players.short_name.eq('H. Kane'), 'player_id'].iloc[0]
# a World Cup match Kane scored in
wc_match_ids = set(matches.loc[matches.competition_id.eq('world_cup'),'match_id'])
kane_pms = pms[(pms.player_id==kane_id) & (pms.match_id.isin(wc_match_ids))]
mid = int(kane_pms.sort_values('goals', ascending=False).match_id.iloc[0])
print('match:', matches.loc[matches.match_id.eq(mid),'label'].iloc[0])

print('\n--- INGREDIENT A: appearance row (minutes + authoritative goals) ---')
display(appearances[(appearances.player_id==kane_id)&(appearances.match_id==mid)]
        [['player_id','match_id','minutes_played','started','goals']])

print('--- INGREDIENT B: his events that match, aggregated ---')
ke = events_wc[(events_wc.player_id==kane_id)&(events_wc.match_id==mid)]
print('total events:', len(ke), '| passes:', int(ke.is_pass.sum()),
      '| shots:', int(ke.is_shot.sum()), '| event goals:', int(ke.event_goal.sum()))

print('\n--- RESULT: the single merged player_match_stats row ---')
kane_pms[(kane_pms.match_id==mid)][['player_id','match_id','minutes_played',
    'goals','event_goals','pass_total','shots_total','total_events']]


match: England - Panama, 6 - 1

--- INGREDIENT A: appearance row (minutes + authoritative goals) ---


,player_id,match_id,minutes_played,started,goals
72731,8717,2057993,63.0,True,3


--- INGREDIENT B: his events that match, aggregated ---
total events: 24 | passes: 12 | shots: 1 | event goals: 3

--- RESULT: the single merged player_match_stats row ---


,player_id,match_id,minutes_played,goals,event_goals,pass_total,shots_total,total_events
52549,8717,2057993,63.0,3,3,12,1,24


## Why so many tables? (the takeaway)

Each table is one clean, testable step — and each is reused by many things:

- **`players_master` / `teams_master`** — small lookup tables joined on by id for
  names & positions. Kept separate so bio data lives in one place.
- **`matches_master`** — match context (scores, coaches). Reused by careers
  (goals conceded) *and* coaches (results).
- **`player_appearances`** — the ONLY trustworthy source of minutes. Split out so
  the fragile minutes logic is isolated and validated on its own.
- **`events_enriched`** — huge (millions of rows); saved per competition so it
  never all sits in memory at once, and so the expensive flagging runs only once.
- **`player_match_stats`** — the atomic join everything downstream is built from.

Next stop: **Phase 4** rolls `player_match_stats` up into one row per player
(`player_career_stats`) — per-90 rates, accuracy %, GK save%.
